In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ephys.src.utils.utils_IO import fetch_good_units, fetch_session_events
from ephys.src.utils.utils_analysis import (
    compute_population_peth,
    compute_unit_selectivity,
    classify_peak_count,
)

%matplotlib widget

In [ ]:
subject = "GRB058"
session = "20260224_152424"

st_per_unit = fetch_good_units(subject, session)
align_ev = fetch_session_events(subject, session)

unit_ids = list(st_per_unit.keys())
print(
    f"Loaded {len(unit_ids)} units, {len(align_ev['first_stim_ev'])} first-stim events"
)

In [ ]:
# Compute population PETH aligned to first stimulus onset
peth, bin_edges, bin_centers = compute_population_peth(
    spike_times_per_unit=list(st_per_unit.values()),
    alignment_times=align_ev["first_stim_ev"],
    pre_seconds=0.1,
    post_seconds=0.15,
    binwidth_ms=10,
    t_rise=0.001,
    t_decay=0.025,
)
print(f"PETH shape: {peth.shape}  (units, trials, timebins)")

In [ ]:
# Baseline vs response selectivity test
results_df, masks = compute_unit_selectivity(
    peth,
    bin_edges,
    unit_ids=unit_ids,
    base_window=(-0.04, 0.0),
    resp_window=(0.06, 0.10),
    test="wilcoxon",
    correction="bonferroni",
    alpha=0.05,
)

n_units = peth.shape[0]
print(
    f"Selective: {masks['selective'].sum()}/{n_units} "
    f"({masks['excited'].sum()} excited, {masks['suppressed'].sum()} suppressed)"
)
results_df.head(10)

In [ ]:
# Classify excited units by number of PSTH peaks
# (suppressed units have sub-baseline responses — peak detection doesn't apply)
exc_idx = np.where(masks["excited"])[0]
exc_peth = peth[exc_idx]
exc_ids = [unit_ids[i] for i in exc_idx]

peaks_df = classify_peak_count(
    exc_peth,
    bin_centers,
    unit_ids=exc_ids,
    search_window=(0.0, 0.15),
    baseline_window=(-0.04, 0.0),
    min_prominence_frac=0.25,
    min_distance_ms=20.0,
    binwidth_ms=10.0,
)

# Merge peak info into results_df
results_df = results_df.merge(
    peaks_df[["unit", "n_peaks", "peak_times"]], on="unit", how="left"
)
results_df["n_peaks"] = results_df["n_peaks"].fillna(0).astype(int)

print(f"Peak count distribution ({len(exc_ids)} excited units):")
print(peaks_df["n_peaks"].value_counts().sort_index().to_string())
print("\nSample rows:")
peaks_df.head(10)

In [ ]:
# Example PSTHs grouped by peak count, with detected peaks marked
n_examples = 3  # examples per group
peak_groups = sorted(peaks_df["n_peaks"].unique())
peak_groups = [g for g in peak_groups if g > 0]  # skip 0-peak units

fig, axes = plt.subplots(
    len(peak_groups),
    n_examples,
    figsize=(3.5 * n_examples, 3 * len(peak_groups)),
    sharey=False,
    squeeze=False,
)

for row_i, npk in enumerate(peak_groups):
    group = peaks_df[peaks_df["n_peaks"] == npk].head(n_examples)
    for col_j, (_, prow) in enumerate(group.iterrows()):
        ax = axes[row_i, col_j]
        uid = int(prow["unit"])
        pos = exc_ids.index(uid)
        mean_psth = exc_peth[pos].mean(axis=0)

        ax.plot(bin_centers, mean_psth, color="black", linewidth=1.5)
        ax.axvspan(-0.05, 0.0, color="tab:blue", alpha=0.15)
        ax.axvspan(0.06, 0.10, color="tab:orange", alpha=0.15)
        ax.axvline(0, color="red", linestyle="--", linewidth=0.8)

        # Mark detected peaks
        for pt, ph in zip(prow["peak_times"], prow["peak_heights"]):
            ax.plot(pt, ph, "v", color="crimson", markersize=8, zorder=5)

        ax.set(
            xlabel="Time (s)", title=f"Unit {uid} ({npk} peak{'s' if npk > 1 else ''})"
        )
    axes[row_i, 0].set_ylabel("sp/s")

    # blank out unused columns
    for col_j in range(len(group), n_examples):
        axes[row_i, col_j].set_visible(False)

plt.tight_layout()

In [ ]:
# Interactive peak classification browser — all excited units
import ipywidgets as widgets
from IPython.display import display

_peak_color = {0: "#90a4ae", 1: "#ffca28", 2: "#ffa726"}
_peak_label = {0: "0-peak  (unclassified)", 1: "1-peak", 2: "2-peak"}

# Sort by n_peaks so groups are contiguous; include 0-peak units too
_df = peaks_df.sort_values(["n_peaks", "unit"]).reset_index(drop=True)
_n = len(_df)

with plt.ioff():
    _fig, _ax = plt.subplots(figsize=(7, 3.5), dpi=120)


def _plot(idx):
    _ax.clear()
    row = _df.iloc[idx]
    uid, npk = int(row["unit"]), int(row["n_peaks"])
    mean_psth = exc_peth[exc_ids.index(uid)].mean(axis=0)

    _ax.plot(bin_centers, mean_psth, color="black", linewidth=1.5)
    _ax.axvspan(-0.04, 0.0, color="tab:blue", alpha=0.15)
    _ax.axvspan(0.06, 0.10, color="tab:orange", alpha=0.15)
    _ax.axvline(0, color="gray", linestyle="--", linewidth=0.8)
    for pt, ph in zip(row["peak_times"], row["peak_heights"]):
        _ax.plot(pt, ph, "v", color="crimson", markersize=10, zorder=5)

    _ax.set(xlabel="Time (s)", ylabel="Firing rate (sp/s)")
    _ax.set_title(
        f"Unit {uid}   [{idx + 1} / {_n}]   —   {_peak_label.get(npk, f'{npk}-peak')}",
        fontsize=11,
        fontweight="bold",
        color=_peak_color.get(npk, "#ff7043"),
    )
    _fig.tight_layout()
    _fig.canvas.draw_idle()


_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=_n - 1,
    step=1,
    layout=widgets.Layout(width="400px"),
)
_prev_btn = widgets.Button(description="◀", layout=widgets.Layout(width="44px"))
_next_btn = widgets.Button(description="▶", layout=widgets.Layout(width="44px"))
_group_dd = widgets.Dropdown(
    options=[("All", -1)]
    + [
        (f"{v}-peak (n={int((peaks_df['n_peaks'] == v).sum())})", v)
        for v in sorted(_df["n_peaks"].unique())
    ],
    description="Jump to:",
    style={"description_width": "auto"},
    layout=widgets.Layout(width="200px"),
)


def _on_prev(_):
    _slider.value = max(0, _slider.value - 1)


def _on_next(_):
    _slider.value = min(_n - 1, _slider.value + 1)


def _on_group(change):
    v = change["new"]
    if v >= 0:
        first = _df.index[_df["n_peaks"] == v]
        if len(first):
            _slider.value = first[0]


_prev_btn.on_click(_on_prev)
_next_btn.on_click(_on_next)
_slider.observe(lambda ch: _plot(ch["new"]), names="value")
_group_dd.observe(_on_group, names="value")

display(_fig.canvas)
display(widgets.HBox([_prev_btn, _slider, _next_btn, _group_dd]))
_plot(0)

In [ ]:
# Summary: pie chart, delta FR histogram, volcano plot
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# --- Pie chart: population composition ---
pie_labels = ["Excited", "Suppressed", "Non-selective"]
pie_counts = [
    int(masks["excited"].sum()),
    int(masks["suppressed"].sum()),
    n_units - int(masks["selective"].sum()),
]
pie_colors = ["#ef5350", "#42a5f5", "#b0bec5"]
wedges, texts, autotexts = axes[0].pie(
    pie_counts,
    labels=pie_labels,
    colors=pie_colors,
    autopct=lambda p: f"{p:.0f}%\n(n={int(round(p * n_units / 100))})",
    startangle=90,
    textprops={"fontsize": 9},
)
for at in autotexts:
    at.set_fontsize(8)
axes[0].set_title("Population composition")

# --- Delta FR histogram ---
sns.histplot(data=results_df, x="delta", bins=31, ax=axes[1], color="steelblue")
axes[1].set(title="Delta FR (response − baseline)", xlabel="ΔFR (sp/s)", ylabel="Count")

# --- Volcano plot ---
sns.scatterplot(
    data=results_df,
    x="cohen_d",
    y=-np.log10(np.clip(results_df["q"], 1e-12, 1.0)),
    hue="selective",
    palette={True: "crimson", False: "gray"},
    ax=axes[2],
    s=20,
)
axes[2].axhline(-np.log10(0.05), color="k", linestyle=":", alpha=0.4)
axes[2].set(title="Volcano plot", xlabel="Cohen's d", ylabel="-log₁₀(q)")
axes[2].legend(title="Selective", loc="best")
plt.tight_layout()

In [ ]:
# Population classification tree
from matplotlib.patches import FancyBboxPatch

# Gather counts
n_total = n_units
n_sel = int(masks["selective"].sum())
n_ns = n_total - n_sel
n_exc = int(masks["excited"].sum())
n_sup = int(masks["suppressed"].sum())
pk_counts = peaks_df["n_peaks"].value_counts().sort_index()

# Tree layout: (label, count, x, y, parent_xy)
nodes = [
    ("All units", n_total, 0.50, 0.92, None),
    ("Non-selective", n_ns, 0.22, 0.62, (0.50, 0.92)),
    ("Selective", n_sel, 0.72, 0.62, (0.50, 0.92)),
    ("Suppressed", n_sup, 0.55, 0.32, (0.72, 0.62)),
    ("Excited", n_exc, 0.85, 0.32, (0.72, 0.62)),
]

# Excited sub-groups by peak count
pk_vals = sorted(pk_counts.index)
n_pk = len(pk_vals)
pk_spacing = 0.20  # increased from 0.14 to avoid box overlap
if n_pk > 0:
    x_start = 0.85 - pk_spacing * (n_pk - 1) / 2
    for i, npk in enumerate(pk_vals):
        nodes.append(
            (
                f"{npk}-peak",
                int(pk_counts[npk]),
                x_start + i * pk_spacing,
                0.07,
                (0.85, 0.32),
            )
        )

# Category colors
cat_colors = {
    "All units": "#546e7a",
    "Non-selective": "#90a4ae",
    "Selective": "#ff7043",
    "Suppressed": "#42a5f5",
    "Excited": "#ef5350",
}
for npk in pk_vals:
    cat_colors[f"{npk}-peak"] = (
        "#ffca28" if npk == 1 else "#ffa726" if npk == 2 else "#ff7043"
    )

fig, ax = plt.subplots(figsize=(6, 4), dpi=150)
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.axis("off")

box_w, box_h = 0.17, 0.11

for label, count, x, y, parent in nodes:
    pct = 100 * count / n_total
    color = cat_colors.get(label, "#90a4ae")

    # Connector line
    if parent is not None:
        px, py = parent
        ax.plot(
            [px, x],
            [py - box_h / 2, y + box_h / 2],
            color="#78909c",
            linewidth=1.5,
            zorder=0,
        )

    # Box
    rect = FancyBboxPatch(
        (x - box_w / 2, y - box_h / 2),
        box_w,
        box_h,
        boxstyle="round,pad=0.012",
        facecolor=color,
        edgecolor="white",
        linewidth=1.5,
        alpha=0.88,
        zorder=1,
    )
    ax.add_patch(rect)

    # Label + count
    ax.text(
        x,
        y + 0.015,
        label,
        ha="center",
        va="center",
        fontsize=9,
        fontweight="bold",
        color="white",
        zorder=2,
    )
    ax.text(
        x,
        y - 0.023,
        f"n = {count}  ({pct:.0f}%)",
        ha="center",
        va="center",
        fontsize=8,
        color="white",
        zorder=2,
    )

ax.set_title(
    f"Population classification — {subject} {session}",
    fontsize=11,
    fontweight="bold",
    pad=10,
)
plt.tight_layout()